In [1]:
#Execute this
#Step1
import pandas as pd
import pandasql as ps
import pixiedust
import sys
sys.path.append('..')
import util
import etl
import pyarrow.parquet as pq
import pyarrow as pa
!pip install duckdb
import duckdb

util.usedatabase(spark, "real_world_data_jun_2022")

pixiedust.enableJobMonitor()

con = duckdb.connect()

Pixiedust database opened successfully


Please see https://github.com/pypa/pip/issues/5599 for advice on fixing the underlying issue.
To avoid this problem you can invoke Python with '-m pip' instead of running pip directly.
Using real_world_data_jun_2022 ....
Successfully enabled Spark Job Progress Monitor


In [1]:
#Execute this
#Total TBI and EPI Extracted with all the TBI and EPI codes
TBI_with_EPI_Total_Final = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/TBI_EPI_Total_Edited')

AnalysisException: 'Path does not exist: file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/TBI_EPI_Total_Edited;'

In [3]:
#Execute this
TBI_with_EPI_Total_Final.createOrReplaceTempView('TBI_EPI_Total')

▸,:,


In [4]:
TBI_with_EPI_Total_Final.printSchema()

▸,:,


root
 |-- personid: string (nullable = true)
 |-- encounterid: string (nullable = true)
 |-- date: string (nullable = true)
 |-- conditioncode: string (nullable = true)



In [4]:
#Execute this
#New Medical codes added- Finalized
epi_med = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/epi_med_Edited')

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
#Execute this
epi_med.createOrReplaceTempView('EPI_Medication_Total')

▸,:,


In [6]:
#Execute this
epi_med.printSchema()

▸,:,


root
 |-- encounterid: string (nullable = true)
 |-- personid: string (nullable = true)
 |-- drugid: string (nullable = true)
 |-- drugname: string (nullable = true)
 |-- startdate: string (nullable = true)
 |-- stopdate: string (nullable = true)



In [7]:
#Execute this
#Only TBI Patients who also can have EPI with unfiltered dates
TBI_Total_Final = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/TBI_Total_Edited')

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
#Execute this
TBI_Total_Final.createOrReplaceTempView('TBI_total')

▸,:,


In [9]:
TBI_Total_Final.printSchema()

▸,:,


root
 |-- personid: string (nullable = true)
 |-- date: string (nullable = true)



In [10]:
#Execute this
#Only EPI Patients
EBI_Total_Final = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/EPI_Total_Edited')

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
#Execute this
EBI_Total_Final.createOrReplaceTempView('EPI_total')

▸,:,


In [12]:
EBI_Total_Final.printSchema()

▸,:,


root
 |-- personid: string (nullable = true)
 |-- date: string (nullable = true)



In [ ]:
# EPI_Test_Empty = spark.sql("""
#   SELECT t.personid, t.date
#   FROM EPI_total t
#   WHERE t.date = '' 
# """)
EPI_Test_NULL = spark.sql("""
  SELECT t.personid, t.date
  FROM EPI_total t
  WHERE t.date IS NULL
""")

# EPI_Test_Empty.show(5, truncate = False)
EPI_Test_NULL.show(5, truncate = False)
# EPI_Test_NotEmpty = spark.sql("""
#   SELECT *
#   FROM EPI_total t
#   WHERE t.personid = '1bc202ff-a7b5-4d15-b5fd-98dee850a7b7'
# """)

# EPI_Test_NotEmpty.show(truncate = False)
# EPI_Test_Duplicates = spark.sql("""
#   SELECT t.personid, COUNT(*) AS count
#   FROM EPI_total t group by t.personid
#   having count >1
# """)

# EPI_Test_Duplicates.show(truncate = False)

In [16]:
#Checking for Empty values in TBI and EPI extracted Parquet
TBI_EPI_Test_NULL = spark.sql("""
  SELECT *
  FROM TBI_EPI_Total t
  WHERE t.date = ''
""")

TBI_EPI_Test_NULL.show(truncate = False)
TBI_EPI_Test_Empty = spark.sql("""
  SELECT *
  FROM TBI_EPI_Total t
  WHERE t.personid = '1bc202ff-a7b5-4d15-b5fd-98dee850a7b7'
""")

TBI_EPI_Test_Empty.show(truncate = False)

+------------------------------------+------------------------------------+----+-------------+
|personid                            |encounterid                         |date|conditioncode|
+------------------------------------+------------------------------------+----+-------------+
|db6d0578-71ab-40a1-8cce-4012c28d683e|01a47470-101b-4519-9d13-92b5bba40a99|    |S06.4X9A     |
|033179c6-d8bb-4088-a646-2bfc75b2c09d|6e14e177-548f-44c3-9108-62f3c5327cb8|    |S06.5X9A     |
|0a5e099d-9702-4e25-8ebb-4f2897d8661a|684ecf85-6074-4c24-9e9c-ff21d3c55c0f|    |Z87.820      |
|d372a639-8e24-48c8-9adc-2e0223b4d348|cceea3f9-118a-4a01-9a77-f41cdca4928c|    |S06.5X9A     |
|10ac6ad0-9da0-4bcb-a50d-4edf91732def|cdf3aab2-d10f-4ceb-bd50-3e12b7e7a872|    |G40.909      |
|5d106077-6ecf-43fe-9070-dd64ca6ad600|e8114619-1032-4b4e-9b35-edfd48965f20|    |R56.9        |
|3c9b88e0-5b0d-4154-94dc-026109f0513f|b50d9f04-a74d-4f67-9a3a-623adafd77c4|    |G40.909      |
|3c9b88e0-5b0d-4154-94dc-026109f0513f|f58d7269-e4b

In [ ]:
# TBI_Test_Empty = spark.sql("""
#   SELECT t.personid, t.date
#   FROM TBI_total t
#   WHERE t.date = ''
# """)

# TBI_Test_Empty.show(5, truncate = False)
# TBI_Test_NotEmpty = spark.sql("""
#   SELECT *
#   FROM TBI_total t
#   WHERE t.personid = '000c88af-2520-4a78-8fca-6c76a485ded2'
# """)

# TBI_Test_NotEmpty.show(truncate = False)
TBI_Test_Duplicates = spark.sql("""
  SELECT t.personid, COUNT(*) AS count
  FROM TBI_total t group by t.personid
  having count >1
""")

TBI_Test_Duplicates.show(truncate = False)

In [17]:
TBI_EPI_Test_Empty = spark.sql("""
  SELECT *
  FROM TBI_EPI_Total t
  WHERE t.personid = '000c88af-2520-4a78-8fca-6c76a485ded2'
""")

TBI_EPI_Test_Empty.show(truncate = False)

+------------------------------------+------------------------------------+-------------------------+-------------+
|personid                            |encounterid                         |date                     |conditioncode|
+------------------------------------+------------------------------------+-------------------------+-------------+
|000c88af-2520-4a78-8fca-6c76a485ded2|4b428c86-6dca-46bb-bb48-0e081ea4614b|                         |S09.90XA     |
|000c88af-2520-4a78-8fca-6c76a485ded2|4b428c86-6dca-46bb-bb48-0e081ea4614b|2021-08-12T22:37:00+00:00|S09.90XA     |
+------------------------------------+------------------------------------+-------------------------+-------------+



In [13]:
#Execute this
# Extract records where TBI_date < EPI_date, or either EPI date is NULL(This can be checked if the patient has dates in EPI Med table)
common_persons_with_date_condition = spark.sql("""
  SELECT 
    T.personid,
    T.date as TBI_date,
    E.date as EPI_date
  FROM 
    TBI_Total T
  INNER JOIN 
    EPI_Total E
  ON 
    T.personid = E.personid
  WHERE 
    (T.date < E.date OR E.date IS NULL)
    AND T.date IS NOT NULL
""")

# Show the output
common_persons_with_date_condition.show(5, truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+-------------------------+-------------------------+
|personid                            |TBI_date                 |EPI_date                 |
+------------------------------------+-------------------------+-------------------------+
|001e4ea5-2b81-4093-a5f2-228c5f56bd19|2016-10-31T04:00:00+00:00|2020-10-25T03:06:00+00:00|
|0137680e-a2a0-4531-bdff-f43052865b70|2020-09-30T19:16:00+00:00|2020-11-04T20:03:00+00:00|
|031f3391-6f93-4ff3-bb68-c6eaa5dbf1a5|2017-11-25T07:00:00+00:00|null                     |
|032145cc-8b8d-41b0-8b15-198fa7c95a68|2012-11-07T06:00:00+00:00|2016-05-13T06:00:00+00:00|
|04569e0a-242b-490d-b948-cd819155dbc2|2009-10-22T00:00:00      |2009-11-02T00:00:00      |
+------------------------------------+-------------------------+-------------------------+
only showing top 5 rows



<IPython.core.display.Javascript object>

In [14]:
#Execute this
common_persons_with_date_condition.createOrReplaceTempView('TBI_With_EPI')

▸,:,


In [15]:
#Testcase
# Extract records where TBI_date is greater than EPI_date from TBI_With_EPI table
filtered_records = spark.sql("""
  SELECT 
    *
  FROM 
    TBI_With_EPI
  WHERE 
    TBI_date > EPI_date or TBI_date IS NULL
""")

# Show the output
filtered_records.show(5, truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+--------+--------+--------+
|personid|TBI_date|EPI_date|
+--------+--------+--------+
+--------+--------+--------+



<IPython.core.display.Javascript object>

In [16]:
#Execute this
# Replace empty date with NULL in Medical table and extract min date
EPI_MED_Total = spark.sql("""
  SELECT 
    personid,
    MIN(CASE WHEN startdate = '' THEN NULL ELSE startdate END) as date
  FROM 
    EPI_Medication_Total 
  GROUP BY 
    personid
""")

# Show the output
EPI_MED_Total.show(5, truncate=False)
EPI_MED_Total.createOrReplaceTempView('EPI_Med_Min')
EPI_Med_Min_CNT = spark.sql("SELECT personid, COUNT(*) as count FROM EPI_Med_Min group by personid having count>1")
EPI_Med_Min_CNT.show()

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+--------+-----+
|personid|count|
+--------+-----+
+--------+-----+



<IPython.core.display.Javascript object>

In [28]:
#Execute this
# Extract records based on your specific conditions(selecting least date between TBIhavingEPI(based on condition code)and EPI Medical table)
# Also Ensuring Not Nulls from both table dates and picking the ones that has values in either of the table dates
common_persons_Diag_Med = spark.sql("""
  SELECT 
    T.personid, T.TBI_date,
    LEAST(COALESCE(T.EPI_date, E.date), COALESCE(E.date, T.EPI_date)) as date
  FROM 
    TBI_With_EPI T
  INNER JOIN 
    EPI_Med_Min E
  ON 
    T.personid = E.personid
  WHERE 
    (T.EPI_date < E.date OR E.date < T.EPI_date OR T.EPI_date IS NOT NULL OR E.date IS NOT NULL)
    AND NOT (T.EPI_date IS NULL AND E.date IS NULL)
""")

# Show the output
common_persons_Diag_Med.show(5, truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+-------------------------+-------------------------+
|personid                            |TBI_date                 |date                     |
+------------------------------------+-------------------------+-------------------------+
|001e4ea5-2b81-4093-a5f2-228c5f56bd19|2016-10-31T04:00:00+00:00|2014-05-23T14:18:00+00:00|
|0137680e-a2a0-4531-bdff-f43052865b70|2020-09-30T19:16:00+00:00|2020-10-01T02:50:00+00:00|
|031f3391-6f93-4ff3-bb68-c6eaa5dbf1a5|2017-11-25T07:00:00+00:00|2017-02-13T00:17:00+00:00|
|032145cc-8b8d-41b0-8b15-198fa7c95a68|2012-11-07T06:00:00+00:00|2012-07-24T21:04:16+00:00|
|04860a66-a13b-45bd-a0a1-a4d476cec2a1|2021-03-25T22:34:00+00:00|2016-08-26T14:12:49+00:00|
+------------------------------------+-------------------------+-------------------------+
only showing top 5 rows



<IPython.core.display.Javascript object>

In [35]:
#Execute this 
common_persons_Diag_Med.createOrReplaceTempView('TBI_EPI_MED_Cohort')
#TestCases
# TBI_EPI_MED_Cohort_Check_2 = spark.sql("""common_persons_Diag_Med
#   SELECT * 
#   FROM TBI_EPI_MED_Cohort 
#   WHERE personid IN ('031f3391-6f93-4ff3-bb68-c6eaa5dbf1a5', '0137680e-a2a0-4531-bdff-f43052865b70', '0907ca1e-d377-45c6-b9dc-f41cf71adf35', '2a6145c2-b763-48f7-89c6-bd18a726a7b1', '0e354c03-9193-449d-a6a3-e762efc4d82f', '2cb70d0f-7e0f-4d7f-87d5-fb89c87577cc')
# """)
# TBI_EPI_MED_Cohort_Check_2.show(truncate=False)
# TBI_EPI_MED_Cohort_Check_1 = spark.sql("""
#   SELECT t.personid, t.EPI_date, e.date 
#   FROM TBI_With_EPI t 
#   INNER JOIN EPI_Med_Min e 
#   ON t.personid = e.personid 
#   WHERE t.personid IN ('031f3391-6f93-4ff3-bb68-c6eaa5dbf1a5', '0137680e-a2a0-4531-bdff-f43052865b70', '0907ca1e-d377-45c6-b9dc-f41cf71adf35', '2a6145c2-b763-48f7-89c6-bd18a726a7b1', '0e354c03-9193-449d-a6a3-e762efc4d82f', '2cb70d0f-7e0f-4d7f-87d5-fb89c87577cc')
# """)

# TBI_EPI_MED_Cohort_Check_1.show(truncate=False)

▸,:,


In [34]:
common_persons_Diag_Med.printSchema()

▸,:,


root
 |-- personid: string (nullable = true)
 |-- TBI_date: string (nullable = true)
 |-- date: string (nullable = true)



In [36]:
Testing_TBIDate = spark.sql("""
  SELECT * from TBI_EPI_MED_Cohort
""")
Testing_TBIDate.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+-------------------------+-------------------------+
|personid                            |TBI_date                 |date                     |
+------------------------------------+-------------------------+-------------------------+
|001e4ea5-2b81-4093-a5f2-228c5f56bd19|2016-10-31T04:00:00+00:00|2014-05-23T14:18:00+00:00|
|0137680e-a2a0-4531-bdff-f43052865b70|2020-09-30T19:16:00+00:00|2020-10-01T02:50:00+00:00|
|031f3391-6f93-4ff3-bb68-c6eaa5dbf1a5|2017-11-25T07:00:00+00:00|2017-02-13T00:17:00+00:00|
|032145cc-8b8d-41b0-8b15-198fa7c95a68|2012-11-07T06:00:00+00:00|2012-07-24T21:04:16+00:00|
|04860a66-a13b-45bd-a0a1-a4d476cec2a1|2021-03-25T22:34:00+00:00|2016-08-26T14:12:49+00:00|
|04c5063f-7ff4-418c-b3c7-bb0d8dd5595a|2018-01-13T03:33:00+00:00|2009-10-13T17:26:00+00:00|
|055bfe77-885b-438b-bb0e-9673cb93bebb|2017-12-15T16:55:00+00:00|2013-05-08T05:16:02+00:00|
|05d7bd53-b21b-4147-99a9-ecbfbcdcefd9|2017-11-02T05:00:00+00:00|2012-01-11T17:10:39+00:00|

<IPython.core.display.Javascript object>

In [ ]:
TBI_EPI_MED_Cohort_Check_3 = spark.sql("""
  SELECT personid, COUNT(*) as count
  FROM TBI_EPI_MED_Cohort 
  GROUP BY personid 
  HAVING count > 1
""")
TBI_EPI_MED_Cohort_Check_3.show(truncate=False)

In [ ]:
#Cohort Based on both TBI having EPI based on cond.code and Medical Table but final one is in the below cell
common_persons_Diag_Med.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/TBI_EPI_MED_Cohort_Edited_V1")

In [19]:
#Directly Execute this
TBI_EPI_MED_Cohort_Load = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/TBI_EPI_MED_Cohort_Edited_V1')

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
TBI_EPI_MED_Cohort_Check_1 = spark.sql("SELECT t.personid, t.EPI_date, e.date FROM TBI_With_EPI t INNER JOIN EPI_Med_Min e on t.personid = e.personid where WHERE personid IN ('031f3391-6f93-4ff3-bb68-c6eaa5dbf1a5', '0137680e-a2a0-4531-bdff-f43052865b70', '0907ca1e-d377-45c6-b9dc-f41cf71adf35', '2a6145c2-b763-48f7-89c6-bd18a726a7b1', '0e354c03-9193-449d-a6a3-e762efc4d82f', '2cb70d0f-7e0f-4d7f-87d5-fb89c87577cc')")
TBI_EPI_MED_Cohort_Check_1.show(20,truncate=False)

In [ ]:
TBI_EPI_MED_Cohort_Check = spark.sql("SELECT personid, COUNT(*) as count FROM TBI_EPI_MED_Cohort group by personid having count>1")
TBI_EPI_MED_Cohort_Check.show(truncate=False)

In [ ]:
#Control Prep- only TBI-> Exclude TBI_date greater than EPI_date Records
# Extract records where TBI_date is greater than EPI_date from TBI_With_EPI table
filtered_records = spark.sql("""
  SELECT 
    *
  FROM 
    TBI_With_EPI
  WHERE 
    TBI_date = '' or EPI_date = ''
""")

# Show the output
filtered_records.show(5, truncate=False)

In [ ]:
# Checking Condition table for dates other than effective date in order to 
filtered_dates = spark.sql("""
  SELECT effectivedate, asserteddate FROM condition WHERE personid = '0174463f-a826-49e1-93da-89968c106c80'
""")

# Show the output
filtered_dates.show(truncate=False)

In [ ]:
# Checking 'TBI_EPI_Total_Edited' table 'TBI_EPI_Total' for dates from etl.py with distinct
filtered_dates = spark.sql("""
  SELECT date FROM TBI_EPI_Total WHERE ((personid = '0174463f-a826-49e1-93da-89968c106c80') and (conditioncode in ('Z87.820','S02.1','R56.1') or
conditioncode like 'S06%' or
conditioncode like 'S07%' or
conditioncode like 'S08%' or
conditioncode like 'S09%' or
conditioncode like 'G44.3%' or
conditioncode like '854.%' or
conditioncode like '851.%' or
conditioncode like '852.%' or
conditioncode like '853.%'))
""")

# Show the output
filtered_dates.show(truncate=False)

In [ ]:
# Checking 'TBI_EPI_Total_Edited' table 'TBI_EPI_Total' for dates from etl.py with distinct
filtered_dates = spark.sql("""
  SELECT date FROM TBI_EPI_Total WHERE ((personid = '0174463f-a826-49e1-93da-89968c106c80') and (conditioncode in ('780.39','R55','780.2','Q04.3','742.4') or
conditioncode like 'G40%' or
conditioncode like '345%' or
conditioncode like 'R56.%'))
""")

# Show the output
filtered_dates.show(truncate=False)

In [ ]:
# Checking 'TBI_EPI_Total_Edited' table 'TBI_EPI_Total' for dates from etl.py with distinct
filtered_dates = spark.sql("""
  SELECT date FROM TBI_EPI_Total WHERE personid = '0174463f-a826-49e1-93da-89968c106c80'
""")

# Show the output
filtered_dates.show(20,truncate=False)

In [ ]:
# Checking NO nulls dates for TBI
filtered_dates = spark.sql("""
  SELECT personid, EPI_date, TBI_date FROM TBI_With_EPI where EPI_date IS NULL
""")

# Show the output
filtered_dates.show(20,truncate=False)

In [23]:
#Execute this
# Control Prep - pnly TBI and not having EPI
# Extract records where TBI NOT in EPI and TBI- Diag date is not NULL
TBI_SUBTRACT_EPI = spark.sql("""
  SELECT 
    T.personid,
    T.date as TBI_date
  FROM 
    TBI_Total T
  LEFT OUTER JOIN 
    EPI_Total E
  ON 
    T.personid = E.personid
  WHERE 
    E.personid IS NULL
    AND T.date IS NOT NULL
""")

# Show the output
TBI_SUBTRACT_EPI.show(5, truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+-------------------------+
|personid                            |TBI_date                 |
+------------------------------------+-------------------------+
|0004c82c-aed4-4726-8efa-abb3edfbba69|2018-10-16T23:44:00+00:00|
|00223bb6-9f64-4410-bc6d-8698235e9c59|2020-10-17T00:00:00      |
|003d713a-a3e1-434a-bec1-01a8a0f4bc40|2018-09-23T07:00:00+00:00|
|0040fc74-4065-450c-9f15-c271b79023df|2019-11-14T05:00:00+00:00|
|0053f0b8-4a78-4826-b193-9a92cd4e43d5|2009-08-13T17:37:44+00:00|
+------------------------------------+-------------------------+
only showing top 5 rows



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
#Execute this 
#TBI which are not present in EPI cond.code but present in Medical
# Based on TBI_date<EPI and TBI is not NULL and EPI is not NULL
TBI_SUBTRACT_EPI.createOrReplaceTempView('TBI_WithOUT_EPI')
# Extract records based on your specific conditions
common_persons_Diag_Med_1 = spark.sql("""
  SELECT 
    T.personid,
    T.TBI_date,
    E.date
  FROM 
    TBI_WithOUT_EPI T
  INNER JOIN 
    EPI_Med_Min E
  ON 
    T.personid = E.personid
  WHERE 
    T.TBI_date < E.date
    AND T.TBI_date IS NOT NULL
    AND E.date IS NOT NULL
""")

# Show the output
common_persons_Diag_Med_1.show(5, truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+-------------------------+-------------------------+
|personid                            |TBI_date                 |date                     |
+------------------------------------+-------------------------+-------------------------+
|0053f0b8-4a78-4826-b193-9a92cd4e43d5|2009-08-13T17:37:44+00:00|2015-05-13T12:11:00+00:00|
|010f943d-aae3-42b4-b175-190914c45e7d|2020-11-22T19:26:00+00:00|2022-03-18T11:49:00+00:00|
|0139734b-df33-4803-957d-ce25f551ee08|2019-08-18T19:46:00+00:00|2020-02-19T11:58:57+00:00|
|021f919e-3d29-47bd-a987-217bc23773a1|2018-09-21T04:00:00+00:00|2018-09-26T17:31:43+00:00|
|02515c4b-bcad-49bd-b5a5-ed43af4b7b59|2019-05-18T04:00:00+00:00|2021-06-08T14:26:03+00:00|
+------------------------------------+-------------------------+-------------------------+
only showing top 5 rows



<IPython.core.display.Javascript object>

In [25]:
#Execute this
common_persons_Diag_Med_1.createOrReplaceTempView('TBI_WithOUT_EPI_with_EPIMed')
Test_1 = spark.sql("""
  SELECT *
  FROM 
    TBI_WithOUT_EPI_with_EPIMed T
  WHERE 
    T.TBI_date > T.date
    OR T.TBI_date IS NULL
    OR T.date IS NULL
""")

# Show the output
Test_1.show(5, truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+--------+--------+----+
|personid|TBI_date|date|
+--------+--------+----+
+--------+--------+----+



<IPython.core.display.Javascript object>

In [ ]:
common_persons_query = spark.sql("""
  SELECT T.*, C.*
  FROM TBI_EPI_MED_Cohort C
  INNER JOIN TBI_WithOUT_EPI_with_EPIMed T
  ON C.personid = T.personid
""")

# Show the output
common_persons_query.show(5, truncate=False)

In [37]:
#Execute this
# Create a new DataFrame without the TBI_date column
# new_df_TBI_EPI_MED_Cohort = spark.sql("SELECT * FROM TBI_EPI_MED_Cohort").drop('TBI_date')
#Wrote this code to include TBI_date for pairing process
new_df_TBI_EPI_MED_Cohort = spark.sql("SELECT * FROM TBI_EPI_MED_Cohort")
# If you want to overwrite the existing table
new_df_TBI_EPI_MED_Cohort.createOrReplaceTempView("TBI_EPI_MED_Cohort_V1")

# Show the DataFrame to verify the column has been removed
new_df_TBI_EPI_MED_Cohort.show(5, truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+-------------------------+-------------------------+
|personid                            |TBI_date                 |date                     |
+------------------------------------+-------------------------+-------------------------+
|001e4ea5-2b81-4093-a5f2-228c5f56bd19|2016-10-31T04:00:00+00:00|2014-05-23T14:18:00+00:00|
|0137680e-a2a0-4531-bdff-f43052865b70|2020-09-30T19:16:00+00:00|2020-10-01T02:50:00+00:00|
|031f3391-6f93-4ff3-bb68-c6eaa5dbf1a5|2017-11-25T07:00:00+00:00|2017-02-13T00:17:00+00:00|
|032145cc-8b8d-41b0-8b15-198fa7c95a68|2012-11-07T06:00:00+00:00|2012-07-24T21:04:16+00:00|
|04860a66-a13b-45bd-a0a1-a4d476cec2a1|2021-03-25T22:34:00+00:00|2016-08-26T14:12:49+00:00|
+------------------------------------+-------------------------+-------------------------+
only showing top 5 rows



<IPython.core.display.Javascript object>

In [38]:
new_df_TBI_EPI_MED_Cohort.printSchema()

▸,:,


root
 |-- personid: string (nullable = true)
 |-- TBI_date: string (nullable = true)
 |-- date: string (nullable = true)



In [39]:
# # Create a new DataFrame without the TBI_date column
# new_df_TBI_WithOUT_EPI_with_EPIMed_Cohort = spark.sql("SELECT * FROM TBI_WithOUT_EPI_with_EPIMed").drop('TBI_date')

#Editing to include TBI Date for pairing
# Create a new DataFrame without the TBI_date column
new_df_TBI_WithOUT_EPI_with_EPIMed_Cohort = spark.sql("SELECT * FROM TBI_WithOUT_EPI_with_EPIMed")

# If you want to overwrite the existing table
new_df_TBI_WithOUT_EPI_with_EPIMed_Cohort.createOrReplaceTempView("TBI_EPI_MED_Cohort_V2")

# Show the DataFrame to verify the column has been removed
new_df_TBI_WithOUT_EPI_with_EPIMed_Cohort.show(5, truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+-------------------------+-------------------------+
|personid                            |TBI_date                 |date                     |
+------------------------------------+-------------------------+-------------------------+
|0053f0b8-4a78-4826-b193-9a92cd4e43d5|2009-08-13T17:37:44+00:00|2015-05-13T12:11:00+00:00|
|010f943d-aae3-42b4-b175-190914c45e7d|2020-11-22T19:26:00+00:00|2022-03-18T11:49:00+00:00|
|0139734b-df33-4803-957d-ce25f551ee08|2019-08-18T19:46:00+00:00|2020-02-19T11:58:57+00:00|
|021f919e-3d29-47bd-a987-217bc23773a1|2018-09-21T04:00:00+00:00|2018-09-26T17:31:43+00:00|
|02515c4b-bcad-49bd-b5a5-ed43af4b7b59|2019-05-18T04:00:00+00:00|2021-06-08T14:26:03+00:00|
+------------------------------------+-------------------------+-------------------------+
only showing top 5 rows



<IPython.core.display.Javascript object>

In [40]:
#Execute this
new_df_TBI_EPI_MED_Cohort.printSchema()
new_df_TBI_WithOUT_EPI_with_EPIMed_Cohort.printSchema()

▸,:,


root
 |-- personid: string (nullable = true)
 |-- TBI_date: string (nullable = true)
 |-- date: string (nullable = true)

root
 |-- personid: string (nullable = true)
 |-- TBI_date: string (nullable = true)
 |-- date: string (nullable = true)



In [41]:
# Stacking the two DataFrames
stacked_df = new_df_TBI_EPI_MED_Cohort.union(new_df_TBI_WithOUT_EPI_with_EPIMed_Cohort)

# Show the resulting DataFrame
stacked_df.show(5, truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+-------------------------+-------------------------+
|personid                            |TBI_date                 |date                     |
+------------------------------------+-------------------------+-------------------------+
|001e4ea5-2b81-4093-a5f2-228c5f56bd19|2016-10-31T04:00:00+00:00|2014-05-23T14:18:00+00:00|
|0137680e-a2a0-4531-bdff-f43052865b70|2020-09-30T19:16:00+00:00|2020-10-01T02:50:00+00:00|
|031f3391-6f93-4ff3-bb68-c6eaa5dbf1a5|2017-11-25T07:00:00+00:00|2017-02-13T00:17:00+00:00|
|032145cc-8b8d-41b0-8b15-198fa7c95a68|2012-11-07T06:00:00+00:00|2012-07-24T21:04:16+00:00|
|04860a66-a13b-45bd-a0a1-a4d476cec2a1|2021-03-25T22:34:00+00:00|2016-08-26T14:12:49+00:00|
+------------------------------------+-------------------------+-------------------------+
only showing top 5 rows



<IPython.core.display.Javascript object>

In [42]:
stacked_df.createOrReplaceTempView("TBI_EPI_MED_Cohort_FinalVersion")
Test_Version = spark.sql("""
  SELECT personid, COUNT(*) as count
  FROM TBI_EPI_MED_Cohort_FinalVersion group by personid having count>1
""")
Test_Version.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+--------+-----+
|personid|count|
+--------+-----+
+--------+-----+



<IPython.core.display.Javascript object>

In [43]:
stacked_df = stacked_df.withColumnRenamed("date", "EPI_date")

▸,:,


In [44]:
#Finalized - Cohort -> TBI Having EPI-> Rewritting to Add TBI_Date to Disease Group
stacked_df.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/TBI_Having_Epilepsy_Cohort_Add_TBIDate")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#Finalized - Cohort -> TBI Having EPI
stacked_df.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/TBI_Having_Epilepsy_Cohort")

In [18]:
# Prepare Conrol Group-> TBI not present in EPI and also not present in EPI Med table
# This excludes records having EPI date first and then TBI Dates as well
# Contains only TBI patients
# TBI_Only = spark.sql("""
#     SELECT *
#     FROM TBI_WithOUT_EPI
#     WHERE personid NOT IN (SELECT personid FROM EPI_Med_Min)
# """)
# TBI_Only.show(5)
TBI_Only = spark.sql("""
    SELECT A.*
    FROM TBI_WithOUT_EPI A
    LEFT ANTI JOIN EPI_Med_Min B
    ON A.personid = B.personid
""")
TBI_Only.show(5, truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+--------------------+--------------------+
|            personid|            TBI_date|
+--------------------+--------------------+
|0004c82c-aed4-472...|2018-10-16T23:44:...|
|00223bb6-9f64-441...| 2020-10-17T00:00:00|
|003d713a-a3e1-434...|2018-09-23T07:00:...|
|0040fc74-4065-450...|2019-11-14T05:00:...|
|0077e451-d2d7-401...| 2016-12-05T00:00:00|
+--------------------+--------------------+
only showing top 5 rows



<IPython.core.display.Javascript object>

In [19]:
TBI_Only.show(5, truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+-------------------------+
|personid                            |TBI_date                 |
+------------------------------------+-------------------------+
|0004c82c-aed4-4726-8efa-abb3edfbba69|2018-10-16T23:44:00+00:00|
|00223bb6-9f64-4410-bc6d-8698235e9c59|2020-10-17T00:00:00      |
|003d713a-a3e1-434a-bec1-01a8a0f4bc40|2018-09-23T07:00:00+00:00|
|0040fc74-4065-450c-9f15-c271b79023df|2019-11-14T05:00:00+00:00|
|0077e451-d2d7-4018-a0d3-9b882c0b17a0|2016-12-05T00:00:00      |
+------------------------------------+-------------------------+
only showing top 5 rows



<IPython.core.display.Javascript object>

In [20]:
TBI_Only_Test = spark.sql("""
    SELECT * FROM EPI_Med_Min where personid IN ('0004c82c-aed4-4726-8efa-abb3edfbba69', '00223bb6-9f64-4410-bc6d-8698235e9c59', '003d713a-a3e1-434a-bec1-01a8a0f4bc40', '0040fc74-4065-450c-9f15-c271b79023df', '0077e451-d2d7-4018-a0d3-9b882c0b17a0')
""")
TBI_Only_Test.show(5)
TBI_Only.createOrReplaceTempView("TBI_Control")
TBI_Only_Test_1 = spark.sql("""
    SELECT personid, COUNT(*) as count FROM TBI_Control group by personid having count>1
""")
TBI_Only_Test_1.show(5)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+--------+-----+
|personid|count|
+--------+-----+
+--------+-----+



<IPython.core.display.Javascript object>

In [21]:
TBI_Only.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/TBI_NotHaving_Epilepsy_Control")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [46]:
TBI_Only = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/TBI_NotHaving_Epilepsy_Control')

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [47]:
TBI_Only.printSchema()

▸,:,


root
 |-- personid: string (nullable = true)
 |-- TBI_date: string (nullable = true)



In [2]:
TBI_with_EPI_Total_Final = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/TBI_EPI_Total_Edited_v1')

In [3]:
TBI_with_EPI_Total_Final.printSchema()

root
 |-- personid: string (nullable = true)
 |-- encounterid: string (nullable = true)
 |-- date: string (nullable = true)
 |-- conditioncode: string (nullable = true)



In [4]:
TBI_with_EPI_Total_Final.show(5, truncate=False)

+------------------------------------+------------------------------------+-------------------------+-------------+
|personid                            |encounterid                         |date                     |conditioncode|
+------------------------------------+------------------------------------+-------------------------+-------------+
|3389feeb-d859-4dc7-9b2b-6c0fda896677|6746cb1d-90d7-4285-a839-46a7f8fdf2ad|2019-03-30T07:00:00+00:00|S09.8XXA     |
|dfb489d3-7359-45cc-82f5-f50c22c50da1|74cce0bf-864a-4fde-8d6c-8ecc8820b46f|2017-06-16T07:00:00+00:00|S09.90XA     |
|16f6a129-c401-4b31-9b1c-bc324d7f06af|561da668-d750-4ace-b489-987ce608787a|2019-02-01T08:00:00+00:00|R55          |
|217a5af3-b4c1-4a78-8194-ff96522d5720|fbb56ad2-74e4-4626-b67e-a03c4a4a9c79|2020-08-05T19:00:00+00:00|R56.9        |
|217a5af3-b4c1-4a78-8194-ff96522d5720|493d70d2-ee92-4fb2-bed7-bb83a9cf20a1|2021-04-18T19:00:00+00:00|R56.9        |
+------------------------------------+----------------------------------

In [6]:
print(TBI_with_EPI_Total_Final.distinct().count())
print(TBI_with_EPI_Total_Final.select("personid").distinct().count())

15407228
4591860


In [20]:
TBI_with_EPI_Total_Final.createOrReplaceTempView('TBI_EPI_Test')
# TBI_EPI_Cohort_Test = spark.sql("""
#   SELECT * 
#   FROM TBI_EPI_Test   
#   WHERE personid IN ('031f3391-6f93-4ff3-bb68-c6eaa5dbf1a5', '0137680e-a2a0-4531-bdff-f43052865b70', '0907ca1e-d377-45c6-b9dc-f41cf71adf35', '2a6145c2-b763-48f7-89c6-bd18a726a7b1', '0e354c03-9193-449d-a6a3-e762efc4d82f', '2cb70d0f-7e0f-4d7f-87d5-fb89c87577cc')
# """)
# TBI_EPI_Cohort_Test.show(truncate=False)
TBI_EPI_Cohort_Test = spark.sql("""
  SELECT * 
  FROM TBI_EPI_Test   
  WHERE personid = '031f3391-6f93-4ff3-bb68-c6eaa5dbf1a5' """)
TBI_EPI_Cohort_Test.show(truncate=False)

+------------------------------------+------------------------------------+-------------------------+-------------+
|personid                            |encounterid                         |date                     |conditioncode|
+------------------------------------+------------------------------------+-------------------------+-------------+
|031f3391-6f93-4ff3-bb68-c6eaa5dbf1a5|a1d5d352-feaf-4e6b-a383-b0c5268867c3|                         |S09.90XD     |
|031f3391-6f93-4ff3-bb68-c6eaa5dbf1a5|a1d5d352-feaf-4e6b-a383-b0c5268867c3|2017-11-25T07:00:00+00:00|Z87.820      |
|031f3391-6f93-4ff3-bb68-c6eaa5dbf1a5|324652be-d474-4ae2-8eee-fbaa7b8e4279|                         |R55          |
|031f3391-6f93-4ff3-bb68-c6eaa5dbf1a5|324652be-d474-4ae2-8eee-fbaa7b8e4279|                         |R55          |
+------------------------------------+------------------------------------+-------------------------+-------------+



In [21]:
# TBI_EPI_Cohort_Test = spark.sql("""
#   SELECT * 
#   FROM TBI_EPI_Total 
#   WHERE personid IN ('031f3391-6f93-4ff3-bb68-c6eaa5dbf1a5', '0137680e-a2a0-4531-bdff-f43052865b70', '0907ca1e-d377-45c6-b9dc-f41cf71adf35', '2a6145c2-b763-48f7-89c6-bd18a726a7b1', '0e354c03-9193-449d-a6a3-e762efc4d82f', '2cb70d0f-7e0f-4d7f-87d5-fb89c87577cc')
# """)
# TBI_EPI_Cohort_Test.show(truncate=False)
TBI_EPI_Cohort_Test_1 = spark.sql("""
  SELECT * 
  FROM TBI_EPI_Total 
  WHERE personid = '031f3391-6f93-4ff3-bb68-c6eaa5dbf1a5' """)
TBI_EPI_Cohort_Test_1.show(truncate=False)

+------------------------------------+------------------------------------+-------------------------+-------------+
|personid                            |encounterid                         |date                     |conditioncode|
+------------------------------------+------------------------------------+-------------------------+-------------+
|031f3391-6f93-4ff3-bb68-c6eaa5dbf1a5|324652be-d474-4ae2-8eee-fbaa7b8e4279|                         |R55          |
|031f3391-6f93-4ff3-bb68-c6eaa5dbf1a5|a1d5d352-feaf-4e6b-a383-b0c5268867c3|2017-11-25T07:00:00+00:00|Z87.820      |
|031f3391-6f93-4ff3-bb68-c6eaa5dbf1a5|a1d5d352-feaf-4e6b-a383-b0c5268867c3|                         |S09.90XD     |
+------------------------------------+------------------------------------+-------------------------+-------------+



In [10]:
TBI_EPI_Cohort_Test_Final = spark.sql("""
SELECT DISTINCT T.*, E.startdate
FROM TBI_EPI_Total T
INNER JOIN EPI_Medication_Total E
ON T.personid = E.personid
WHERE T.date IS NOT NULL
  AND E.startdate IS NOT NULL
  """)
TBI_EPI_Cohort_Test_Final.show(5, truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+------------------------------------+-------------------------+-------------+-------------------------+
|personid                            |encounterid                         |date                     |conditioncode|startdate                |
+------------------------------------+------------------------------------+-------------------------+-------------+-------------------------+
|000218b2-8b69-40a9-a70a-db8bab5d747d|04c83510-b636-4970-a409-a50d66990b40|2018-04-13T08:58:00+00:00|R55          |2011-01-08T11:57:00+00:00|
|000218b2-8b69-40a9-a70a-db8bab5d747d|04c83510-b636-4970-a409-a50d66990b40|2018-04-13T08:58:00+00:00|R55          |2011-01-08T22:54:00+00:00|
|000218b2-8b69-40a9-a70a-db8bab5d747d|04c83510-b636-4970-a409-a50d66990b40|2018-04-13T08:58:00+00:00|R55          |2011-01-08T00:46:00+00:00|
|000218b2-8b69-40a9-a70a-db8bab5d747d|04c83510-b636-4970-a409-a50d66990b40|2018-04-13T08:58:00+00:00|R55          |2022-06-01T20:37:00+00:00|
|00021

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
Cohort_Read = spark.read.parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/TBI_Having_Epilepsy_Cohort')
Cohort_Read.createOrReplaceTempView('Cohort')

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [21]:
TBI_EPI_Cohort_Test_final_Check = spark.sql("""
  SELECT * 
  FROM Cohort 
  WHERE personid = '0029fb87-5cc9-417b-860d-4889dec9ea2b' """)
TBI_EPI_Cohort_Test_final_Check.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+--------+----+
|personid|date|
+--------+----+
+--------+----+



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
TBI_EPI_Cohort_Test_final_Check1 = spark.sql("""
  SELECT * 
  FROM TBI_total 
  WHERE personid = '000218b2-8b69-40a9-a70a-db8bab5d747d' """)
TBI_EPI_Cohort_Test_final_Check1.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+--------+----+
|personid|date|
+--------+----+
+--------+----+



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
TBI_EPI_Cohort_Test_final_Check2 = spark.sql("""
  SELECT * 
  FROM EPI_total 
  WHERE personid = '000218b2-8b69-40a9-a70a-db8bab5d747d' """)
TBI_EPI_Cohort_Test_final_Check2.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+-------------------------+
|personid                            |date                     |
+------------------------------------+-------------------------+
|000218b2-8b69-40a9-a70a-db8bab5d747d|2018-04-12T07:00:00+00:00|
+------------------------------------+-------------------------+



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
TBI_EPI_Cohort_Test_Final1 = spark.sql("""
SELECT DISTINCT T.personid
FROM TBI_total T
INNER JOIN EPI_total E
ON T.personid = E.personid
WHERE T.date IS NOT NULL
  AND E.date IS NOT NULL
  """)
TBI_EPI_Cohort_Test_Final1.show(5, truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+
|personid                            |
+------------------------------------+
|001e4ea5-2b81-4093-a5f2-228c5f56bd19|
|0029fb87-5cc9-417b-860d-4889dec9ea2b|
|00717999-29ac-478f-95d0-7da8afa92ddc|
|009c62ad-bdd6-49bc-8f4a-c5f69fbf16b4|
|0137680e-a2a0-4531-bdff-f43052865b70|
+------------------------------------+
only showing top 5 rows



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
TBI_EPI_Cohort_Test_final_Check1 = spark.sql("""
  SELECT * 
  FROM TBI_total 
  WHERE personid = '0029fb87-5cc9-417b-860d-4889dec9ea2b' """)
TBI_EPI_Cohort_Test_final_Check1.show(truncate=False)
TBI_EPI_Cohort_Test_final_Check2 = spark.sql("""
  SELECT * 
  FROM EPI_total 
  WHERE personid = '0029fb87-5cc9-417b-860d-4889dec9ea2b' """)
TBI_EPI_Cohort_Test_final_Check2.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+-------------------------+
|personid                            |date                     |
+------------------------------------+-------------------------+
|0029fb87-5cc9-417b-860d-4889dec9ea2b|2022-07-03T19:12:00+00:00|
+------------------------------------+-------------------------+



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [25]:
#TestCase
Cohort_Check_1 = spark.sql("""
SELECT DISTINCT T.personid
FROM TBI_total T
INNER JOIN EPI_total E
ON T.personid = E.personid
INNER JOIN EPI_Medication_Total M
ON T.personid = M.personid
WHERE T.date < E.date
  AND T.date < M.startdate """)
Cohort_Check_1.show(5, truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+
|personid                            |
+------------------------------------+
|001e4ea5-2b81-4093-a5f2-228c5f56bd19|
|0137680e-a2a0-4531-bdff-f43052865b70|
|032145cc-8b8d-41b0-8b15-198fa7c95a68|
|04c5063f-7ff4-418c-b3c7-bb0d8dd5595a|
|055bfe77-885b-438b-bb0e-9673cb93bebb|
+------------------------------------+
only showing top 5 rows



<IPython.core.display.Javascript object>

In [26]:
TBI_EPI_Cohort_Test_final_Check = spark.sql("""
  SELECT * 
  FROM Cohort 
  WHERE personid = '001e4ea5-2b81-4093-a5f2-228c5f56bd19' """)
TBI_EPI_Cohort_Test_final_Check.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+-------------------------+
|personid                            |date                     |
+------------------------------------+-------------------------+
|001e4ea5-2b81-4093-a5f2-228c5f56bd19|2014-05-23T14:18:00+00:00|
+------------------------------------+-------------------------+



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [27]:
TBI_EPI_Cohort_Test_final_Check1 = spark.sql("""
  SELECT * 
  FROM TBI_total 
  WHERE personid = '001e4ea5-2b81-4093-a5f2-228c5f56bd19' """)
TBI_EPI_Cohort_Test_final_Check1.show(truncate=False)
TBI_EPI_Cohort_Test_final_Check2 = spark.sql("""
  SELECT * 
  FROM EPI_total 
  WHERE personid = '001e4ea5-2b81-4093-a5f2-228c5f56bd19' """)
TBI_EPI_Cohort_Test_final_Check2.show(truncate=False)
TBI_EPI_Cohort_Test_final_Check3 = spark.sql("""
  SELECT * 
  FROM EPI_Medication_Total 
  WHERE personid = '001e4ea5-2b81-4093-a5f2-228c5f56bd19' """)
TBI_EPI_Cohort_Test_final_Check3.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+------------------------------------+------+------------------------------+-------------------------+-------------------------+
|encounterid                         |personid                            |drugid|drugname                      |startdate                |stopdate                 |
+------------------------------------+------------------------------------+------+------------------------------+-------------------------+-------------------------+
|5238202d-921e-4f83-8e39-ca6e6bd6b24f|001e4ea5-2b81-4093-a5f2-228c5f56bd19|d03182|gabapentin                    |2019-04-09T18:56:00+00:00|2019-04-09T18:56:00+00:00|
|cc353149-b811-4e74-b5c2-8d8bf6c6fa26|001e4ea5-2b81-4093-a5f2-228c5f56bd19|d00301|midazolam                     |2019-04-09T10:55:00+00:00|2019-04-09T10:55:00+00:00|
|c69838b2-e4db-481b-99b1-73e660d0e0a5|001e4ea5-2b81-4093-a5f2-228c5f56bd19|19429 |gabapentin 300 mg oral capsule|2014-05-23T14:18:00+00:00|2014-12-23T21:46:35+00:00|
|7e5

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>